In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
from pathlib import Path

DATA_DIR = Path('final_results_copy')

In [ ]:

# ── Algorithm name normalisation ─────────────────────────────────────────────
# Strip sequence suffixes so variants collapse to one row, then take the mean.

SEQ_PATTERN = re.compile(
    r'\s*\((water only|water|fat fraction|fat_fraction|dixon|Dixon|both channels?|both)\)\s*$',
    re.IGNORECASE
)

def base_name(name: str) -> str:
    return SEQ_PATTERN.sub('', name).strip()

def extract_seq(name):
    m = re.search(r'\((water only|water|fat fraction|dixon|both[^)]*?)\)', name, re.IGNORECASE)
    if m:
        s = m.group(1).strip().lower()
        return {'water': 'Water', 'water only': 'Water',
                'fat fraction': 'Fat fraction',
                'dixon': 'Dixon'}.get(s, s.capitalize())
    return 'Single'

# Algorithm display order (top = best overall)
ALG_ORDER = [
    'MuscleMap WB',
    'MuscleMap Thigh',
    'MM WB + MedSAM bbox',
    'MM WB + SLM-SAM2',
    'MuSeg',
    'Hirriririir',
    'MedCLIP-SAMv2 Text+Boxes',
    'MM WB + MedSAM mask',
    'MedCLIP-SAMv2',
    'Dafne + MedSAM',
    'Dafne',
    'MedSegDiff',
]

# Algorithm family → label  (aligned with Table 1)
FAMILY = {
    'MuscleMap WB':             'U-Net',
    'MuscleMap Thigh':          'U-Net',
    'MuSeg':                    'nnU-Net',
    'Hirriririir':              'SegResNet (MONAI)',
    'MM WB + MedSAM bbox':      'SAM-based',
    'MM WB + MedSAM mask':      'SAM-based',
    'MM WB + SLM-SAM2':         'SAM-based',
    'MedCLIP-SAMv2':            'SAM-based',
    'MedCLIP-SAMv2 Text+Boxes': 'SAM-based',
    'Dafne + MedSAM':           'Federated DL',
    'Dafne':                    'Federated DL',
    'MedSegDiff':               'Diffusion',
}

FAMILY_COLORS = {
    'U-Net':            '#1f77b4',
    'nnU-Net':          '#aec7e8',
    'SegResNet (MONAI)':'#9467bd',
    'SAM-based':        '#ff7f0e',
    'Federated DL':     '#2ca02c',
    'Diffusion':        '#d62728',
}

DATASETS = {
    'MyoSegmenTUM': 'overall_means_myosegmentum.csv',
    'Pathological': 'overall_means_P_only.csv',
    'AIPS':         'overall_means_asian.csv',
    'Sheffield':    'overall_means_sheffield.csv',
    'Augmented':    'overall_means_augmented.csv',
}

SEQ_ORDER   = ['Water', 'Fat fraction', 'Dixon', 'Single']
SEQ_HATCHES = {'Water': '', 'Fat fraction': '///', 'Dixon': 'xxx', 'Single': ''}


In [ ]:
# ── Load CSVs and build aggregated wide matrices ──────────────────────────────

frames = {}
for ds_name, fname in DATASETS.items():
    df = pd.read_csv(DATA_DIR / fname)
    df['algorithm'] = df['algorithm'].apply(base_name)
    df = df.groupby('algorithm', as_index=False)[['dice', 'hausdorff']].mean()
    frames[ds_name] = df.set_index('algorithm')

dice_wide = pd.DataFrame({ds: frames[ds]['dice']      for ds in DATASETS}).T
hd_wide   = pd.DataFrame({ds: frames[ds]['hausdorff'] for ds in DATASETS}).T

present   = [a for a in ALG_ORDER if a in dice_wide.columns]
extra     = sorted([a for a in dice_wide.columns if a not in present])
col_order = present + extra

dice_wide = dice_wide[col_order]
hd_wide   = hd_wide[col_order]

print('Dice matrix shape:', dice_wide.shape)
dice_wide.round(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Dual heatmap: Dice (left) + Hausdorff (right)
# ═══════════════════════════════════════════════════════════════════════════════

n_alg = dice_wide.shape[1]
n_ds  = dice_wide.shape[0]

fig, axes = plt.subplots(1, 2,
    figsize=(max(14, n_alg * 1.1), max(4, n_ds * 0.9 + 1.2)),
    gridspec_kw={'wspace': 0.45})

# --- Dice ---
ax = axes[0]
im = ax.imshow(dice_wide.values.astype(float), aspect='auto',
               cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(n_alg))
ax.set_xticklabels(dice_wide.columns, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(n_ds))
ax.set_yticklabels(dice_wide.index, fontsize=9)
ax.set_title('Dice Score (\u2191)', fontsize=12, fontweight='bold', pad=10)
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
for i in range(n_ds):
    for j in range(n_alg):
        val = dice_wide.iloc[i, j]
        if pd.isna(val):
            ax.text(j+0.5, i+0.5, '\u2014', ha='center', va='center', fontsize=7, color='#888')
        else:
            tc = 'white' if val < 0.35 else 'black'
            ax.text(j+0.5, i+0.5, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color=tc, fontweight='bold')

# --- Hausdorff ---
ax2 = axes[1]
hd_clip = hd_wide.clip(upper=300)
im2 = ax2.imshow(hd_clip.values.astype(float), aspect='auto',
                 cmap='RdYlGn_r', vmin=0, vmax=300)
ax2.set_xticks(range(n_alg))
ax2.set_xticklabels(hd_wide.columns, rotation=40, ha='right', fontsize=8)
ax2.set_yticks(range(n_ds))
ax2.set_yticklabels(hd_wide.index, fontsize=9)
ax2.set_title('Hausdorff Distance mm (\u2193, clipped 300 mm)', fontsize=12, fontweight='bold', pad=10)
plt.colorbar(im2, ax=ax2, fraction=0.025, pad=0.02)
for i in range(n_ds):
    for j in range(n_alg):
        val = hd_wide.iloc[i, j]
        if pd.isna(val):
            ax2.text(j+0.5, i+0.5, '\u2014', ha='center', va='center', fontsize=6, color='#888')
        else:
            tc = 'white' if hd_clip.iloc[i, j] > 200 else 'black'
            ax2.text(j+0.5, i+0.5, f'{val:.0f}', ha='center', va='center',
                     fontsize=6, color=tc, fontweight='bold')

fig.suptitle('Algorithm \u00d7 Dataset Performance Heatmaps\n(values averaged across MRI sequences)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('heatmap_dice_hausdorff.pdf', bbox_inches='tight')
plt.savefig('heatmap_dice_hausdorff.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved heatmap_dice_hausdorff.pdf / .png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURES 2–6 — Bar chart per dataset: Dice by algorithm, grouped by sequence
# ═══════════════════════════════════════════════════════════════════════════════

for ds_name, fname in DATASETS.items():
    df_raw = pd.read_csv(DATA_DIR / fname)
    df_raw['base']     = df_raw['algorithm'].apply(base_name)
    df_raw['sequence'] = df_raw['algorithm'].apply(extract_seq)

    df_bar = df_raw[df_raw['base'].isin(ALG_ORDER)].copy()
    if df_bar.empty:
        print(f'No matching algorithms for {ds_name}, skipping.')
        continue

    # Sort algorithms by mean Dice on this dataset (descending)
    alg_mean      = df_bar.groupby('base')['dice'].mean()
    alg_order_bar = alg_mean.sort_values(ascending=False).index.tolist()

    seq_present = [s for s in SEQ_ORDER if s in df_bar['sequence'].unique()]
    n_seq       = len(seq_present)
    x           = np.arange(len(alg_order_bar))
    width       = 0.8 / n_seq

    fig, ax = plt.subplots(figsize=(max(10, len(alg_order_bar) * 1.2), 6))

    for k, seq in enumerate(seq_present):
        subset = df_bar[df_bar['sequence'] == seq].set_index('base')['dice']
        vals   = [subset.get(a, np.nan) for a in alg_order_bar]
        colors = [FAMILY_COLORS[FAMILY.get(a, 'U-Net (supervised)')] for a in alg_order_bar]
        offset = (k - n_seq / 2 + 0.5) * width
        ax.bar(x + offset, vals, width * 0.92, label=seq,
               color=colors, hatch=SEQ_HATCHES[seq],
               edgecolor='white', linewidth=0.5, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(alg_order_bar, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Dice Score', fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.set_title(f'{ds_name} \u2014 Dice Score by Algorithm and MRI Sequence',
                 fontsize=12, fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)

    # Sequence legend (hatch) — only when >1 sequence present
    if n_seq > 1:
        seq_legend = [mpatches.Patch(facecolor='grey', hatch=SEQ_HATCHES[s],
                                     label=s, alpha=0.85)
                      for s in seq_present]
        l1 = ax.legend(handles=seq_legend, title='Sequence',
                       loc='upper right', framealpha=0.9, fontsize=8)
        ax.add_artist(l1)

    # Algorithm family legend (colour)
    families_present = {FAMILY.get(a) for a in alg_order_bar} - {None}
    fam_legend = [mpatches.Patch(facecolor=FAMILY_COLORS[f], label=f)
                  for f in FAMILY_COLORS if f in families_present]
    ax.legend(handles=fam_legend, title='Algorithm family',
              loc='upper center', bbox_to_anchor=(0.5, -0.25),
              ncol=min(len(fam_legend), 3), fontsize=8, framealpha=0.9)

    plt.tight_layout()
    slug = ds_name.lower().replace(' ', '_')
    plt.savefig(f'bar_{slug}_by_sequence.pdf', bbox_inches='tight')
    plt.savefig(f'bar_{slug}_by_sequence.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved bar_{slug}_by_sequence.pdf / .png')